# 03 · Qualidade do bronze — estoque e financeiro

Terceira e última varredura de qualidade: as três camadas de estoque (movimento, saldo, foto oficial), os lotes e suas validades, e o lado financeiro (faturas e custos) que alimentará o BI de margens. Mesmo método dos notebooks 01 e 02.

**Como rodar:** com o bronze extraído (`uv run bronze-staging`), kernel da `.venv` do projeto.

In [1]:
import pandas as pd

from logistica_fictitur.lake import conectar

pd.set_option("display.max_rows", 60)
conn = conectar()


def sql(consulta: str) -> pd.DataFrame:
    return conn.execute(consulta).df()

## 1. Panorama: as três camadas de estoque e o calendário das fotos

O estoque vive em três registros com papéis distintos (diário de movimentos, posição corrente e foto oficial de cobrança). Primeiro, o censo; depois, o calendário das fotos, que deveria ter um fechamento por mês.

In [2]:
sql("""
    SELECT 'movimento_estoque' AS tabela, count(*) AS linhas FROM bronze_fwm_movimento_estoque
    UNION ALL SELECT 'estoque_snapshot', count(*) FROM bronze_fwm_estoque_snapshot
    UNION ALL SELECT 'estoque_saldo', count(*) FROM bronze_fwm_estoque_saldo
    UNION ALL SELECT 'lote', count(*) FROM bronze_fwm_lote
    UNION ALL SELECT 'fatura_frete', count(*) FROM bronze_faturamento_fatura_frete
    UNION ALL SELECT 'fatura_armazenagem', count(*) FROM bronze_faturamento_fatura_armazenagem
    UNION ALL SELECT 'custo_operacao', count(*) FROM bronze_financeiro_custo_operacao
    ORDER BY linhas DESC
""")

,tabela,linhas
0,movimento_estoque,4941831
1,fatura_frete,1030210
2,custo_operacao,922313
3,estoque_snapshot,159314
4,lote,10509
5,fatura_armazenagem,9234
6,estoque_saldo,1983


In [3]:
sql("""
    SELECT count(DISTINCT data) AS fotos_distintas,
           min(data) AS primeira, max(data) AS ultima,
           count(DISTINCT date_trunc('month', data)) AS meses_cobertos
    FROM bronze_fwm_estoque_snapshot
""")

,fotos_distintas,primeira,ultima,meses_cobertos
0,81,2020-01-28,2026-09-01,81


> **Nota Técnica 1**  
>**Observado:** as três camadas existem e o calendário de fotos cobre a operação inteira, um fechamento por mês mais o parcial do mês corrente, sem buraco.
> - **Por que importa:** a foto mensal é a base da cobrança de armazenagem; um mês sem foto seria um mês sem faturamento auditável.
> - **Ação proposta:** a checagem de cobertura mensal entra na régua permanente.

## 2. Validades de lote: o absurdo e o vencido

Duas doenças distintas: validade **impossível** (décadas no futuro, defeito de dado) e lote **vencido ainda com presença na última foto** (problema de negócio verdadeiro, não de dado).

In [4]:
sql("""
    SELECT 'lotes com validade absurda (alem de 2040)' AS verificacao,
           count(*) AS ocorrencias
    FROM bronze_fwm_lote WHERE dt_validade > DATE '2040-01-01'
    UNION ALL
    SELECT 'lotes vencidos (validade anterior a data corrente do caso)',
           count(*)
    FROM bronze_fwm_lote WHERE dt_validade < DATE '2026-09-01'
""")

,verificacao,ocorrencias
0,lotes com validade absurda (alem de 2040),47
1,lotes vencidos (validade anterior a data corre...,624


In [5]:
sql("""
    WITH ultima_foto AS (
        SELECT item_id, sum(qtde_saldo) AS saldo
        FROM bronze_fwm_estoque_snapshot
        WHERE data = (SELECT max(data) FROM bronze_fwm_estoque_snapshot)
        GROUP BY 1 HAVING sum(qtde_saldo) > 0
    )
    SELECT count(DISTINCT l.item_id) AS itens_com_lote_vencido_e_presenca_na_foto
    FROM bronze_fwm_lote l
    JOIN ultima_foto u USING (item_id)
    WHERE l.dt_validade < DATE '2026-09-01'
""")

,itens_com_lote_vencido_e_presenca_na_foto
0,209


> **Nota Técnica 2**  
>**Observado:** um punhado de validades impossíveis (décadas à frente, o velho erro de digitação de ano) e uma população relevante de lotes vencidos cujos itens seguem presentes na foto mais recente.
> - **Por que importa:** são tratamentos opostos: a validade absurda é defeito de DADO (a silver anula e marca); o vencido com presença é achado de NEGÓCIO, exatamente o tipo de alerta que o BI de estoque existe para dar, e não se "limpa", se reporta.
> - **Ação proposta:** `fl_validade_invalida` para o absurdo; o vencido permanece intacto e vira indicador na gold.

## 3. O razão fecha com a foto? (conciliação movimentos × snapshot)

Teste de conciliação clássico: somando entradas e saídas do diário de movimentos, o saldo teórico de cada item deveria se aproximar da foto oficial e jamais ser negativo.

In [6]:
sql("""
    WITH razao AS (
        SELECT item_id, sum(quantidade) AS saldo_teorico
        FROM bronze_fwm_movimento_estoque
        GROUP BY 1
    )
    SELECT count(*) AS itens_movimentados,
           count(*) FILTER (WHERE saldo_teorico < 0) AS com_saldo_teorico_negativo,
           round(100.0 * count(*) FILTER (WHERE saldo_teorico < 0) / count(*), 1)
               AS pct_negativos
    FROM razao
""")

,itens_movimentados,com_saldo_teorico_negativo,pct_negativos
0,5286,49,0.9


In [7]:
sql("""
    WITH razao AS (
        SELECT item_id, sum(quantidade) AS saldo_teorico
        FROM bronze_fwm_movimento_estoque GROUP BY 1
    ),
    foto AS (
        SELECT item_id, sum(qtde_saldo) AS saldo_foto
        FROM bronze_fwm_estoque_snapshot
        WHERE data = (SELECT max(data) FROM bronze_fwm_estoque_snapshot)
        GROUP BY 1
    )
    SELECT count(*) AS itens_comparaveis,
           count(*) FILTER (WHERE r.saldo_teorico <> f.saldo_foto) AS divergentes,
           round(100.0 * count(*) FILTER (WHERE r.saldo_teorico <> f.saldo_foto)
                 / count(*), 1) AS pct_divergentes
    FROM razao r JOIN foto f USING (item_id)
""")

,itens_comparaveis,divergentes,pct_divergentes
0,1983,1983,100.0


> **Nota Técnica 3**  
>**Observado:** o razão de movimentos **não fecha** com a foto oficial, e o retrato tem duas camadas: a divergência é **generalizada** (todo item comparável diverge da foto) e, num grupo pequeno mas impossível, o saldo teórico chega a ficar **negativo** (saíram mais unidades do que o diário registra de entrada).
> - **Por que importa:** é a assinatura do sistema cujo módulo de movimentação não captura tudo (cargas iniciais não registradas, ajustes fora do diário, integrações parciais): quem calcular estoque somando movimentos vai publicar números errados com convicção, e o saldo negativo é a prova de que nem como aproximação o razão serve.
> - **Ação proposta:** decisão de FONTE DA VERDADE: **a foto oficial é a autoridade de saldo e cobrança**; o diário de movimentos fica marcado como não-conciliado (`fl_razao_nao_concilia`) e serve para análise de fluxo relativo, nunca de saldo absoluto. É premissa a validar com o dono do processo, como manda a doutrina.

## 4. Financeiro: fretes simbólicos e o mapa dos custos

Dois exames no lado do dinheiro: faturas de frete com valor simbólico (que destroem qualquer ticket médio) e a cobertura real das categorias de custo previstas no modelo.

In [8]:
sql("""
    SELECT count(*) AS faturas_frete,
           count(*) FILTER (WHERE valor_frete_icms <= 0.05) AS valor_simbolico,
           round(100.0 * count(*) FILTER (WHERE valor_frete_icms <= 0.05) / count(*), 1)
               AS pct_simbolico,
           round(avg(valor_frete_icms), 2) AS ticket_medio_com_simbolicos,
           round(avg(valor_frete_icms) FILTER (WHERE valor_frete_icms > 0.05), 2)
               AS ticket_medio_sem_simbolicos
    FROM bronze_faturamento_fatura_frete
""")

,faturas_frete,valor_simbolico,pct_simbolico,ticket_medio_com_simbolicos,ticket_medio_sem_simbolicos
0,1030210,124108,12.0,3720.56,4230.16


In [9]:
sql("""
    SELECT categoria, count(*) AS lancamentos, round(sum(valor), 2) AS total
    FROM bronze_financeiro_custo_operacao
    GROUP BY 1 ORDER BY 2 DESC
""")

,categoria,lancamentos,total
0,CUSTO_FRETE,906102,1.519343e+09
1,DEV_REENTREGA,16211,1.209634e+07


In [10]:
sql("""
    SELECT 'os' AS tabela_de_servico, count(*) AS linhas FROM bronze_fwm_os
    UNION ALL SELECT 'coleta', count(*) FROM bronze_expedicao_coleta
    UNION ALL SELECT 'positivacao', count(*) FROM bronze_fwm_positivacao
    UNION ALL SELECT 'servico_complementar', count(*) FROM bronze_fwm_servico_complementar
    UNION ALL SELECT 'fatura_servico', count(*) FROM bronze_faturamento_fatura_servico
""")

,tabela_de_servico,linhas
0,os,0
1,coleta,0
2,positivacao,0
3,servico_complementar,0
4,fatura_servico,0


> **Nota Técnica 4**  
>**Observado:** uma fatia relevante das faturas de frete carrega valor simbólico (o centavo de retirada no balcão), e a diferença entre o ticket médio com e sem essas linhas mostra o estrago que elas fazem numa média; no mapa de custos, só duas das seis categorias previstas têm lançamentos, e as cinco tabelas do universo de serviços estão vazias.
> - **Por que importa:** o frete simbólico é o `meta_frete` do dinheiro: convenção de sistema que precisa de marcação explícita; e categorias sem lançamento significam que o BI de margens da v1 enxerga transporte e armazenagem, não o portfólio completo de serviços.
> - **Ação proposta:** `fl_frete_simbolico` na silver (linha preservada, fora dos indicadores de receita média); o vazio dos serviços entra documentado como limite de escopo da base v1, irmão do CAD-01.

## 5. Sanidade das margens no tempo

Fechando com uma leitura de plausibilidade: a margem de contribuição do transporte, ano a ano, calculada das faturas e custos (nunca digitada). Não é análise de negócio ainda, é verificação de que o dado financeiro sustenta uma.

In [11]:
sql("""
    WITH receita AS (
        SELECT extract(year FROM competencia)::int AS ano,
               sum(valor_frete_icms) AS receita,
               sum(valor_frete_icms - valor_frete) AS icms
        FROM bronze_faturamento_fatura_frete GROUP BY 1
    ),
    custo AS (
        SELECT extract(year FROM competencia)::int AS ano, sum(valor) AS cv
        FROM bronze_financeiro_custo_operacao
        WHERE categoria IN ('CUSTO_FRETE', 'DEV_REENTREGA') GROUP BY 1
    )
    SELECT r.ano,
           round(r.receita / 1e6, 2) AS receita_mm,
           round((r.receita - r.receita * 0.0673 - r.icms - c.cv) / r.receita, 3)
               AS mc_pct
    FROM receita r JOIN custo c USING (ano)
    WHERE r.ano BETWEEN 2020 AND 2026
    ORDER BY 1
""")

,ano,receita_mm,mc_pct
0,2020,327.00,0.446
1,2021,431.38,0.447
2,2022,485.17,0.446
3,2023,578.15,0.447
4,2024,676.79,0.447
5,2025,774.47,0.457
6,2026,559.99,0.465


> **Nota Técnica 5**  
>**Observado:** receita crescendo ano a ano e MC% do transporte estável numa faixa plausível ao longo de todo o arco.
> - **Por que importa:** confirma que o financeiro do bronze aguenta o BI de margens que vem na gold; se a MC oscilasse absurdamente, haveria defeito de dado escondido.
> - **Ação proposta:** nenhuma; a taxa de imposto usada aqui vem do parâmetro do caso e a fórmula completa (com vigências) será a da gold.

## 6. Consolidação dos achados do domínio

Os achados de estoque e financeiro, recalculados em célula, fechando a matéria-prima do catálogo da silver.

In [12]:
achados = sql("""
    SELECT 'EST-01' AS codigo, 'lotes com validade absurda (alem de 2040)' AS achado,
           (SELECT count(*) FROM bronze_fwm_lote
            WHERE dt_validade > DATE '2040-01-01') AS medida,
           'anular validade + flag; nao adivinhar' AS acao_proposta
    UNION ALL
    SELECT 'EST-02', 'itens com lote vencido e presenca na ultima foto',
           (SELECT count(DISTINCT l.item_id) FROM bronze_fwm_lote l
            JOIN (SELECT item_id FROM bronze_fwm_estoque_snapshot
                  WHERE data = (SELECT max(data) FROM bronze_fwm_estoque_snapshot)
                  GROUP BY 1 HAVING sum(qtde_saldo) > 0) u USING (item_id)
            WHERE l.dt_validade < DATE '2026-09-01'),
           'achado de NEGOCIO: preservar e reportar na gold'
    UNION ALL
    SELECT 'EST-03', 'itens cujo razao de movimentos nao concilia (saldo teorico negativo)',
           (SELECT count(*) FROM (SELECT item_id FROM bronze_fwm_movimento_estoque
                                  GROUP BY 1 HAVING sum(quantidade) < 0)),
           'foto oficial = fonte da verdade; flag no razao; validar premissa com o dono'
    UNION ALL
    SELECT 'FIN-01', 'faturas de frete com valor simbolico',
           (SELECT count(*) FROM bronze_faturamento_fatura_frete
            WHERE valor_frete_icms <= 0.05),
           'flag; fora dos indicadores de receita media'
    UNION ALL
    SELECT 'FIN-02', 'tabelas do universo de servicos sem nenhuma linha',
           (SELECT count(*) FROM (
               SELECT 1 FROM bronze_fwm_os HAVING count(*) = 0
               UNION ALL SELECT 1 FROM bronze_expedicao_coleta HAVING count(*) = 0
               UNION ALL SELECT 1 FROM bronze_fwm_positivacao HAVING count(*) = 0
               UNION ALL SELECT 1 FROM bronze_fwm_servico_complementar HAVING count(*) = 0
               UNION ALL SELECT 1 FROM bronze_faturamento_fatura_servico
               HAVING count(*) = 0)),
           'limite de escopo da base v1: documentar, nao inventar'
""")
achados

,codigo,achado,medida,acao_proposta
0,EST-01,lotes com validade absurda (alem de 2040),47,anular validade + flag; nao adivinhar
1,EST-02,itens com lote vencido e presenca na ultima foto,209,achado de NEGOCIO: preservar e reportar na gold
2,EST-03,itens cujo razao de movimentos nao concilia (s...,49,foto oficial = fonte da verdade; flag no razao...
3,FIN-01,faturas de frete com valor simbolico,124108,flag; fora dos indicadores de receita media
4,FIN-02,tabelas do universo de servicos sem nenhuma linha,5,"limite de escopo da base v1: documentar, nao i..."


> **Nota Técnica 6 (fecho da EDA)**  
>**Observado:** cinco achados fecham a varredura: dois defeitos de dado (validade absurda, frete simbólico), um problema de conciliação estrutural (razão × foto, com decisão de fonte da verdade a validar), um achado de negócio a preservar (vencidos com presença) e um limite de escopo documentável (serviços vazios).
> - **Por que importa:** somados aos notebooks 01 e 02, o bronze está integralmente auditado: 11 achados no total, cada um com medida executável e ação proposta.
> - **Ação proposta:** consolidar CAD-01..03, PED-01..03 e EST/FIN-01..05 no **catálogo de achados e regras da silver** e submeter à aprovação do dono antes de qualquer transformação.